# Analyizing the NEON Mosquito Data

This notebook assumes you have downloaded the NEON Mosquito data, "Mosquitoes sampled from CO2 Traps", for some sites.
We assume you downloaded the latest and provisional data for two sites, ABBY and UNDE, for 2024 and 2025 where available, as described in the [README](./README.md#neon).
Of course, this code should work for any number of sites and years.

If you haven't already, unzip the data in the week 6 directory.
You will see folder with many nested folders, and inside each of those are many files.
We will focus on the "mos_expertTaxonomistIDProcessed" files, for which there is one for each site for each month.

Our first task is to get all the data loaded.

In [1]:
import pandas as pd
import glob
import datetime
import matplotlib as mp

Our approach to load the data is to first identify all of the relevant CSV files, then load each into its own data frame, and then concatenate the dataframes together.
The glob package will find files based on a pattern.

In [2]:
filenames = glob.glob("NEON_count*/**/*Processed*", recursive=True)
dfs = [pd.read_csv(fname) for fname in filenames]
df = pd.concat(dfs)
print(f"Found {len(dfs)} CSV files")

Found 21 CSV files


We will sort the observations based on the date the samples were collected. To this end we need to make that column a datetime datatype.

In [3]:
df['collectDate'] = pd.to_datetime(df['collectDate'])

And then finally, we only care about the month summary for the data, so make a new column that will let us group on the year/month combination.

In [4]:
df['yearmonth'] = df['collectDate'].values.astype('<M8[M]')

In [5]:
df.set_index('yearmonth', inplace=True)

Each site is indicated by the `siteID` column. This lets us recover the directory structure of the original CSV files.

In [7]:
sites = df.groupby('siteID')

In [ ]:
sites['individualCount'].sum()

Group the recorded individual counts by month/year and site. This sums `individualCount` across the identified taxon/sex records in each site and month. These are counts within the identified subsamples; consult the NEON product documentation before interpreting them as total sample abundance. We are not distinguishing between mosquito species here, though we could group by `scientificName` to compare selected species.

In [12]:
s = df.groupby(['yearmonth','siteID'])['individualCount'].sum().unstack(level=1)

In [ ]:
s.plot()

There looks to be something up with the ABBY data since it is missing July and August 2022 counts.